# 07 - Model Comparison & Benchmarking

**End-to-End Benchmark Analysis**

This notebook computes everything from scratch:
- Loads raw model outputs from `results/final/`
- Computes all metrics, rankings, and statistical tests
- Generates all tables and figures per category and horizon
- Performs cross-horizon analysis
- Creates overall summary

**Coverage:**
- All models (detected dynamically)
- 3 asset categories (crypto, forex, indices)
- 2 horizons (h=4, h=24)
- 12 assets total

**Outputs:**
- Leaderboards per category/horizon
- Metric comparison plots (RMSE, MAE, SMAPE, Directional Accuracy)
- Rank barplots and heatmaps
- Cross-horizon degradation analysis
- Statistical significance tests (Friedman, Diebold-Mariano, Wilcoxon)
- Overall rankings and win count matrices

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display, Markdown

from src.utils.config import ProjectConfig
from src.experiments.benchmark import run_benchmark, _collect_all_results
# Import centralized styling to ensure consistency in notebook plots
from src.evaluation.plots import MODEL_PALETTE, MODEL_ORDER

config = ProjectConfig('..')

print("Configuration loaded successfully!")
print(f"Results directory: {config.get_path('results_dir')}")


## 1. Run Full Benchmark Pipeline

Executes end-to-end benchmarking:
- Collects all aggregated results
- Computes leaderboards and rankings
- Generates all tables and figures
- Runs statistical tests
- Saves outputs to `results/benchmark/`

In [ ]:
# Run full benchmarking pipeline (generates all outputs)

import importlib
import src.evaluation.plots as plots_module
import src.experiments.benchmark as bench_module

# Reload plotting module first (benchmark depends on it)
plots_module = importlib.reload(plots_module)

# Reload benchmark module AFTER plots
bench_module = importlib.reload(bench_module)

print("Running full benchmark pipeline...\n")

try:
    bench_module.run_benchmark(config)
    print("\n✓ Benchmark complete! All outputs saved to results/benchmark/")
except Exception as e:
    print("\n✗ Benchmark failed.")
    raise e

## 2. Collect and Display Results Summary

In [ ]:
# Collect all results
models = list_models()
all_results = _collect_all_results(config, models)

if not all_results.empty:
    print(f'Total result entries: {len(all_results)}')
    print(f'Models: {all_results["model"].unique().tolist()}')
    print(f'Categories: {all_results["category"].unique().tolist()}')
    print(f'Horizons: {all_results["horizon"].unique().tolist()}')
    display(all_results.head(10))
else:
    print('No results available. Run final training first.')

## 3. Leaderboards by Category and Horizon

In [ ]:
if not all_results.empty:
    for category in config.get_categories():
        display(Markdown(f"### {category.upper()}"))
        
        for horizon in config.get_horizons():
            lb_path = config.get_path("results_dir") / "benchmark" / category / str(horizon) / "tables" / "leaderboard.csv"
            if lb_path.exists():
                lb = pd.read_csv(lb_path)
                display(Markdown(f"#### Horizon {horizon}"))
                display(lb[['rank', 'model', 'mean_rank', 'rmse_mean', 'mae_mean', 
                           'directional_accuracy_mean']])
            else:
                print(f"Leaderboard not found for {category} h={horizon}")

## 4. Metric Comparisons

### RMSE, MAE, SMAPE, and Directional Accuracy

In [ ]:
if not all_results.empty:
    metrics_to_plot = ["rmse", "mae", "smape", "directional_accuracy"]
    
    for category in config.get_categories():
        display(Markdown(f"### {category.upper()}"))
        
        for horizon in config.get_horizons():
            display(Markdown(f"#### Horizon {horizon}"))
            
            for metric in metrics_to_plot:
                img_path = config.get_path("results_dir") / "benchmark" / category / str(horizon) / "figures" / f"{metric}_comparison.png"
                if img_path.exists():
                    display(Markdown(f"**{metric.upper()}:**"))
                    display(Image(filename=str(img_path)))
                else:
                    print(f"{metric.upper()} comparison plot not found for {category} h={horizon}")

In [ ]:
if not all_results.empty:
    for category in config.get_categories():
        display(Markdown(f"### {category.upper()}"))
        
        for horizon in config.get_horizons():
            display(Markdown(f"#### Horizon {horizon}"))
            
            # Rank barplot
            barplot_path = config.get_path("results_dir") / "benchmark" / category / str(horizon) / "figures" / "rank_barplot.png"
            if barplot_path.exists():
                display(Markdown("**Model Rankings (Bar Plot):**"))
                display(Image(filename=str(barplot_path)))
            
            # Rank heatmap
            heatmap_path = config.get_path("results_dir") / "benchmark" / category / str(horizon) / "figures" / "rank_heatmap.png"
            if heatmap_path.exists():
                display(Markdown("**Rank Heatmap per Asset:**"))
                display(Image(filename=str(heatmap_path)))

## 5. Rank Visualizations

### Rank Barplots and Heatmaps

## 3. Rank Visualizations

### Rank Barplots and Heatmaps

In [ ]:
if not all_results.empty:
    for category in config.get_categories():
        display(Markdown(f"### {category.upper()}"))
        
        for horizon in config.get_horizons():
            display(Markdown(f"#### Horizon {horizon}"))
            
            # Rank barplot
            barplot_path = config.get_path("results_dir") / "benchmark" / category / str(horizon) / "figures" / "rank_barplot.png"
            if barplot_path.exists():
                display(Markdown("**Model Rankings (Bar Plot):**"))
                display(Image(filename=str(barplot_path)))
            
            # Rank heatmap
            heatmap_path = config.get_path("results_dir") / "benchmark" / category / str(horizon) / "figures" / "rank_heatmap.png"
            if heatmap_path.exists():
                display(Markdown("**Rank Heatmap per Asset:**"))
                display(Image(filename=str(heatmap_path)))

## 6. Cross-Horizon Analysis

### Degradation Curves and Rank Correlation

In [ ]:
if not all_results.empty and len(config.get_horizons()) > 1:
    for category in config.get_categories():
        display(Markdown(f"### {category.upper()}"))
        
        # RMSE vs Horizon
        rmse_path = config.get_path("results_dir") / "benchmark" / category / "cross_horizon" / "figures" / "rmse_vs_horizon.png"
        if rmse_path.exists():
            display(Markdown("**RMSE vs Horizon:**"))
            display(Image(filename=str(rmse_path)))
        
        # Degradation curve
        deg_curve_path = config.get_path("results_dir") / "benchmark" / category / "cross_horizon" / "figures" / "horizon_degradation_curve.png"
        if deg_curve_path.exists():
            display(Markdown("**Degradation Curve (%):**"))
            display(Image(filename=str(deg_curve_path)))
        
        # Rank correlation
        rank_corr_path = config.get_path("results_dir") / "benchmark" / category / "cross_horizon" / "figures" / "rank_correlation.png"
        if rank_corr_path.exists():
            display(Markdown("**Rank Correlation Across Horizons:**"))
            display(Image(filename=str(rank_corr_path)))
        
        # Degradation analysis table
        deg_path = config.get_path("results_dir") / "benchmark" / category / "cross_horizon" / "tables" / "degradation_analysis.csv"
        if deg_path.exists():
            deg_df = pd.read_csv(deg_path)
            display(Markdown("**Degradation Analysis (Table):**"))
            display(deg_df)
        
        # Average rank table
        avg_rank_path = config.get_path("results_dir") / "benchmark" / category / "cross_horizon" / "tables" / "average_rank.csv"
        if avg_rank_path.exists():
            avg_rank_df = pd.read_csv(avg_rank_path, index_col=0)
            display(Markdown("**Average Rank Across Horizons:**"))
            display(avg_rank_df)
        
        # Rank stability table
        stability_path = config.get_path("results_dir") / "benchmark" / category / "cross_horizon" / "tables" / "rank_stability.csv"
        if stability_path.exists():
            stability_df = pd.read_csv(stability_path, index_col=0)
            display(Markdown("**Rank Stability Analysis:**"))
            display(stability_df)

## 7. Statistical Tests

### Friedman, Diebold-Mariano, and Wilcoxon Tests

In [ ]:
if not all_results.empty:
    for category in config.get_categories():
        display(Markdown(f"### {category.upper()}"))
        
        for horizon in config.get_horizons():
            stats_dir = config.get_path("results_dir") / "benchmark" / category / str(horizon) / "statistical_tests"
            
            display(Markdown(f"#### Horizon {horizon}"))
            
            # Friedman test
            friedman_path = stats_dir / "friedman_test.csv"
            if friedman_path.exists():
                friedman_df = pd.read_csv(friedman_path)
                display(Markdown("**Friedman Test:**"))
                display(friedman_df)
            
            # Nemenyi post-hoc test
            nemenyi_path = stats_dir / "nemenyi_test.csv"
            if nemenyi_path.exists():
                nemenyi_df = pd.read_csv(nemenyi_path)
                sig_pairs = nemenyi_df[nemenyi_df['significant_005'] == True]
                if not sig_pairs.empty:
                    display(Markdown("**Nemenyi Post-Hoc Test (Significant Pairs):**"))
                    display(sig_pairs)
                else:
                    print("No significant pairwise differences found (Nemenyi)")
            
            # Diebold-Mariano test
            dm_path = stats_dir / "diebold_mariano.csv"
            if dm_path.exists():
                dm_df = pd.read_csv(dm_path)
                sig_pairs = dm_df[dm_df['significant_005'] == True]
                if not sig_pairs.empty:
                    display(Markdown("**Diebold-Mariano Test (Significant Pairs):**"))
                    display(sig_pairs)
                else:
                    print("No significant pairwise differences found (Diebold-Mariano)")
            
            # Wilcoxon signed-rank test
            wilcoxon_path = stats_dir / "wilcoxon_signed_rank.csv"
            if wilcoxon_path.exists():
                wilcoxon_df = pd.read_csv(wilcoxon_path)
                sig_pairs = wilcoxon_df[wilcoxon_df['significant_005'] == True]
                if not sig_pairs.empty:
                    display(Markdown("**Wilcoxon Signed-Rank Test (Significant Pairs):**"))
                    display(sig_pairs)
                else:
                    print("No significant pairwise differences found (Wilcoxon)")

## 8. Overall Summary

### Overall Rankings, Win Counts, and Metadata

In [ ]:
if not all_results.empty:
    for category in config.get_categories():
        display(Markdown(f"### {category.upper()}"))
        
        overall_dir = config.get_path("results_dir") / "benchmark" / category / "overall_summary"
        
        # Overall ranking
        ranking_path = overall_dir / "overall_ranking.csv"
        if ranking_path.exists():
            ranking_df = pd.read_csv(ranking_path)
            display(Markdown("**Overall Ranking Across Horizons:**"))
            display(ranking_df)
        
        # Win count matrix
        win_path = overall_dir / "win_count_matrix.csv"
        if win_path.exists():
            win_df = pd.read_csv(win_path, index_col=0)
            display(Markdown("**Win Count Matrix (Row beats Column):**"))
            display(win_df)
        
        # Benchmark metadata
        meta_path = overall_dir / "benchmark_metadata.json"
        if meta_path.exists():
            with open(meta_path, 'r') as f:
                meta = json.load(f)
            display(Markdown("**Benchmark Metadata:**"))
            display(Markdown(f"```json\n{json.dumps(meta, indent=2)}\n```"))

display(Markdown("---"))
display(Markdown("## ✓ Benchmarking Complete!"))
display(Markdown("All results saved to `results/benchmark/`"))